El servicio de venta de autos usados Rusty Bargain está desarrollando una aplicación para atraer nuevos clientes. Gracias a esa app, puedes averiguar rápidamente el valor de mercado de tu coche. Tienes acceso al historial: especificaciones técnicas, versiones de equipamiento y precios. Tienes que crear un modelo que determine el valor de mercado.
A Rusty Bargain le interesa:
- la calidad de la predicción;
- la velocidad de la predicción;
- el tiempo requerido para el entrenamiento

Significado de las columnas de los datos:

- DateCrawled — fecha en la que se descargó el perfil de la base de datos
- VehicleType — tipo de carrocería del vehículo
- RegistrationYear — año de matriculación del vehículo
- Gearbox — tipo de caja de cambios
- Power — potencia (CV)
- Model — modelo del vehículo
- Mileage — kilometraje (medido en km de acuerdo con las especificidades regionales del conjunto de datos)
- RegistrationMonth — mes de matriculación del vehículo
- FuelType — tipo de combustible
- Brand — marca del vehículo
- NotRepaired — vehículo con o sin reparación
- DateCreated — fecha de creación del perfil
- NumberOfPictures — número de fotos del vehículo
- PostalCode — código postal del propietario del perfil (usuario)
- LastSeen — fecha de la última vez que el usuario estuvo activo

### Imports

In [ ]:
import os
import pandas as pd
import numpy as np

### Preparación de datos

#### Cargar datos

In [ ]:
nombre_de_archivo = "car_data.csv"
ruta_a_datasets = "datasets"
ruta_completa = os.path.join(ruta_a_datasets, nombre_de_archivo)

In [ ]:
df = pd.read_csv(ruta_completa)

#### Análisis exploratorio de datos (EDA)

In [ ]:
# Ver la información del DataFrame
df.info()

In [ ]:
df.describe()

##### Price

In [ ]:
# Se observan muchos autos con precios inferiores a 30 dólares. Se decide explorar un poco más
df[df["Price"] < 500] # Explorar modelos de autos con precio menor a 500 dólares

In [ ]:
# Se aprecian autos cuyos precios no están acorde a la realidad; ejemplo el Polo de Volkswagen tiene un precio de 300 dólares; aunque tenga 150 mil millas;
# no tiene reparaciones; valorado en el mercado entre 2000 a 3000 dólares (búsqueda de internet). 
# Se decide eliminar autos con precio menor a 500 dólares; para mantener el modelo simple y no tener que hacer una limpieza más profunda de los datos.
df_filtrado = df[df["Price"] >= 500]

In [ ]:
len(df_filtrado)

##### VehicleType

In [ ]:
# Explorar tipos de vehículos.
df_filtrado[df_filtrado["VehicleType"].isna()]

In [ ]:
# Se deciden eliminar todos los vehículos que se desconoce su tipo; porque no se puede inferir el tipo de vehículo y no se puede reemplazar por un valor genérico.
# Además, se sabe que el tipo de vehículo es importante para el precio. Por ejemplo; un SUV es más caro que un sedán.
df_filtrado = df_filtrado[df_filtrado["VehicleType"].notna()]

In [ ]:
len(df_filtrado)

##### Registration Year

In [ ]:
# En la descripción se aprecia que el año de registro tiene un mínimo de 1000 y un máximo de 9999, lo cual no es realista. 
# Se decide eliminar autos cuyo registro sea inferior a 1990 y superior a 2026.
# Los precios de autos varian mucho; los autos viejos y de colección son autos muy caros; al igual que los autos nuevos.
# Para mantener el modelo simple, solo se dejaran autos "modernos" y se eliminarán los autos antíguos; porque se desconoce si son de colección o solo meten ruido al modelo.
df_filtrado = df_filtrado[(df_filtrado["RegistrationYear"] >= 1990) & (df_filtrado["RegistrationYear"] <= 2026)]
df_filtrado["RegistrationYear"].unique()

In [ ]:
len(df_filtrado)

##### Gearbox

In [ ]:
# Explorar Gearbox
df_filtrado["Gearbox"].unique()

In [ ]:
# Ahora se reemplazaran los valores ausentes de la columna "Gearbox" por "manual". 
df_filtrado["Gearbox"] = df_filtrado["Gearbox"].fillna("manual") # Reemplazar valores ausentes por "manual"

In [ ]:
len(df_filtrado)

##### Power

In [ ]:
# Así mismo se observa que hay autos que tiene una potencia (Power) de 0, esto tampoco es realista; se exploran los tipos de autos que tienen potencia menor a 30 
# para decidir si eliminar o reemplazar.
df_filtrado[(df_filtrado["Power"] <30)]["Model"].unique()

In [ ]:
# Se decide eliminar estos registros; hay todo tipo de vehículos en esta lista, carros de golf, camionetas, autos de lujo. Mantener estos registros afectará al modelo.
# En internet se consiguió que los autos promedios tienen una potencia promedio de 100 CV (autos); sin embargo hay autos como el Bettle (Volkswagen) que tiene una potencia de 75 CV, por lo que se decide eliminar autos con potencia de 35 CV.
# Esto coincide con el promedio obtenido en la descripción del DataFrame.
# Se decide dejar solamente autos con potencia mayor o igual a 30 CV
df_filtrado = df_filtrado[df_filtrado["Power"] >= 30]

In [ ]:
len(df_filtrado)

##### Model

In [ ]:
df_filtrado["Model"].unique()

In [ ]:
df_filtrado[df_filtrado["Model"].isna()]

In [ ]:
# Se decide eliminar autos cuyo modelo sea desconocido. El modelo influye mucho en el precio
df_filtrado = df_filtrado[df_filtrado["Model"].notna()]

In [ ]:
len(df_filtrado)

##### Fuel Type

In [ ]:
# Explorar FuelType
df_filtrado["FuelType"].unique()

In [ ]:
# Explorar FuelType
print("Tipos de combustible en el registro de autos:")
print(df_filtrado[df_filtrado["FuelType"].isnull()]["VehicleType"].unique())
print("")
print("Años de registro únicos de autos con FuelType nulo:")
print(df_filtrado[df_filtrado["FuelType"].isnull()]["RegistrationYear"].unique())
print("")    
print("Cantidad de autos con FuelType nulo:")
print(len(df_filtrado[df_filtrado["FuelType"].isnull()]))

In [ ]:
# Se decide eliminar estos registros; ya que es posible que hayan autos híbridos o eléctricos dentro del registro.
df_filtrado = df_filtrado[df_filtrado["FuelType"].notna()] # El tipo de combustible afecta un poco el precio, pero son pocos registros

In [ ]:
len(df_filtrado)

##### Repairs

In [ ]:
df_filtrado["NotRepaired"].isna()

In [ ]:
df_filtrado["NotRepaired"] = df_filtrado["NotRepaired"].fillna("no") # Reemplazar valores ausentes por "No"
df_filtrado["NotRepaired"].unique()

In [ ]:
len(df_filtrado)

In [ ]:
# Ahora con los datos filtrados; se decide dejar unicamente las columnas que aporten información útil para el modelo:
# Price; VehicleType; RegistrationYear; Gearbox; Power; Mileage; FuelType; Brand; NotRepaired

In [ ]:
df_filtrado.info()

## Entrenamiento del modelo 

### Regresion lineal

###  Árbol de decisión 

### Bosques aleatorios (Random Forest)

### XGBoost

## Análisis del modelo

# Lista de control

Escribe 'x' para verificar. Luego presiona Shift+Enter

- [x]  Jupyter Notebook está abierto
- [ ]  El código no tiene errores- [ ]  Las celdas con el código han sido colocadas en orden de ejecución- [ ]  Los datos han sido descargados y preparados- [ ]  Los modelos han sido entrenados
- [ ]  Se realizó el análisis de velocidad y calidad de los modelos